In [2]:
import tensorflow as tf
from tensorflow.keras import layers, models
import numpy as np


In [3]:
num_samples = 1000
num_features = 10
num_classes = 3

In [5]:
x_train = np.random.random((num_samples, num_features)).astype(np.float32)
y_train = np.random.randint(0, num_classes, size=(num_samples,))
y_train = tf.keras.utils.to_categorical(y_train, num_classes)


In [7]:
teacher_model = models.Sequential([
    layers.Dense(128, activation='relu', input_shape=(num_features,)),
    layers.Dense(64, activation='relu'),
    layers.Dense(num_classes, activation='softmax')
])


In [8]:
# Compile the teacher model
teacher_model.compile(optimizer='adam',
                      loss='categorical_crossentropy',
                      metrics=['accuracy'])


In [9]:
teacher_model.fit(x_train, y_train, epochs=5, batch_size=32)


Epoch 1/5
32/32 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.3584 - loss: 1.1013
Epoch 2/5
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.3824 - loss: 1.0876 
Epoch 3/5
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.4309 - loss: 1.0804 
Epoch 4/5
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.4131 - loss: 1.0745 
Epoch 5/5
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.4235 - loss: 1.0753 


In [10]:
student_model = models.Sequential([
    layers.Dense(64, activation='relu', input_shape=(num_features,)),
    layers.Dense(num_classes, activation='softmax')
])


In [11]:
# Compile the student model
student_model.compile(optimizer='adam',
                      loss='categorical_crossentropy',
                      metrics=['accuracy'])


In [12]:
# Define temperature and alpha
temperature = 3.0
alpha = 0.5


In [13]:
# Function for knowledge distillation
def distillation_loss(y_true, y_pred, teacher_output):
    return alpha * tf.keras.losses.categorical_crossentropy(y_true, y_pred) + \
           (1 - alpha) * tf.keras.losses.kullback_leibler_divergence(
               tf.nn.softmax(teacher_output / temperature),
               tf.nn.softmax(y_pred / temperature)
           )


In [14]:
teacher_output = teacher_model.predict(x_train)
student_model.fit(x_train, y_train, epochs=5, batch_size=32,
                  verbose=0,
                  callbacks=[tf.keras.callbacks.LambdaCallback(on_epoch_end=lambda epoch, logs: print(f"Epoch {epoch + 1}: Training student model..."))])


32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step
Epoch 1: Training student model...
Epoch 2: Training student model...
Epoch 3: Training student model...
Epoch 4: Training student model...
Epoch 5: Training student model...


In [15]:
# Step 7: Evaluate both models
teacher_loss, teacher_accuracy = teacher_model.evaluate(x_train, y_train)
student_loss, student_accuracy = student_model.evaluate(x_train, y_train)


32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.4014 - loss: 1.0788  
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.3759 - loss: 1.0904  


In [16]:
print(f"Teacher Model Accuracy: {teacher_accuracy:.4f}")
print(f"Student Model Accuracy: {student_accuracy:.4f}")

Teacher Model Accuracy: 0.4190
Student Model Accuracy: 0.3860
